In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
DATA = ROOT / 'datasets' / 'processed' / 'module1_bus' / 'module1_ml_table.parquet'
MODEL_DIR = ROOT / 'models' / 'module1' / 'baselines'
MET_DIR = ROOT / 'metrics' / 'module1'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MET_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA)
print('rows', len(df))
print(df['priority_action'].value_counts())
print(df.head(3))

Mounted at /content/drive
rows 419453
priority_action
NONE            415421
TRUNCATE_RED      3044
EXTEND_GREEN       988
Name: count, dtype: int64
   route_code direction  node  dist_to_junction_m  speed_mps      eta_sec  \
0         425        up     1         5850.287762        NaN          NaN   
1         425        up     2         5070.043675   0.715820  7082.844589   
2         425        up     3         4782.105986   1.919585  2491.219193   

   lateness_sec  t_sec priority_action  bus_number    month   day  y  \
0      -15539.5  29540            NONE  DL1PC 5373  Month_1  Day1  0   
1      -15144.0  30630            NONE  DL1PC 5373  Month_1  Day1  0   
2      -15962.0  30780            NONE  DL1PC 5373  Month_1  Day1  0   

   direction_up  
0             1  
1             1  
2             1  


In [ ]:
!pip -q install xgboost joblib

In [ ]:
feature_cols = [
    'dist_to_junction_m', 'speed_mps', 'eta_sec', 'lateness_sec',
    't_sec', 'direction_up', 'node', 'route_code'
]

X = df[feature_cols].copy()
# fill remaining NaNs from ETA/speed
X = X.fillna(X.median(numeric_only=True))
y = df['y'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('train', X_train.shape, 'test', X_test.shape)
print('train label counts:\n', y_train.value_counts().sort_index())
print('test label counts:\n', y_test.value_counts().sort_index())

train (335562, 8) test (83891, 8)
train label counts:
 y
0    332337
1       790
2      2435
Name: count, dtype: int64
test label counts:
 y
0    83084
1      198
2      609
Name: count, dtype: int64


In [ ]:
def eval_model(name, model, X_te, y_te):
    pred = model.predict(X_te)
    # probabilities for ROC-AUC (ovr)
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_te)
    else:
        # decision_function fallback
        scores = model.decision_function(X_te)
        if scores.ndim == 1:
            from sklearn.preprocessing import label_binarize
            proba = None
        else:
            # softmax-ish not needed; use sklearn roc on decision scores
            proba = scores

    metrics = {
        'model': name,
        'category': 'Base ML',
        'module': 'module1_bus',
        'accuracy': float(accuracy_score(y_te, pred)),
        'precision': float(precision_score(y_te, pred, average='macro', zero_division=0)),
        'recall': float(recall_score(y_te, pred, average='macro', zero_division=0)),
        'f1': float(f1_score(y_te, pred, average='macro', zero_division=0)),
    }
    try:
        if hasattr(model, 'predict_proba'):
            metrics['roc_auc'] = float(roc_auc_score(y_te, model.predict_proba(X_te), multi_class='ovr', average='macro'))
        else:
            metrics['roc_auc'] = None
    except Exception as e:
        metrics['roc_auc'] = None
        metrics['roc_auc_error'] = str(e)

    print('\n', name)
    print(classification_report(y_te, pred, digits=4, zero_division=0))
    print('confusion_matrix:\n', confusion_matrix(y_te, pred))
    print({k: metrics[k] for k in ['accuracy','precision','recall','f1','roc_auc']})
    return metrics, pred

In [ ]:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        multi_class='auto',
        n_jobs=None
    ))
])
pipe_lr.fit(X_train, y_train)
m0, _ = eval_model('Logistic_Regression', pipe_lr, X_test, y_test)
joblib.dump(pipe_lr, MODEL_DIR / 'logistic_regression.joblib')
(MET_DIR / 'Logistic_Regression.json').write_text(json.dumps(m0, indent=2))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



 Logistic_Regression
              precision    recall  f1-score   support

           0     1.0000    0.9648    0.9821     83084
           1     0.3379    0.9899    0.5039       198
           2     0.1928    0.9967    0.3230       609

    accuracy                         0.9651     83891
   macro avg     0.5102    0.9838    0.6030     83891
weighted avg     0.9926    0.9651    0.9762     83891

confusion_matrix:
 [[80160   384  2540]
 [    0   196     2]
 [    2     0   607]]
{'accuracy': 0.9650975670810933, 'precision': 0.5102218970765494, 'recall': 0.9838072064335804, 'f1': 0.6029921146885607, 'roc_auc': 0.9936500207264004}


250

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=5,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
m1, _ = eval_model('Random_Forest', rf, X_test, y_test)
joblib.dump(rf, MODEL_DIR / 'random_forest.joblib')
(MET_DIR / 'Random_Forest.json').write_text(json.dumps(m1, indent=2))


 Random_Forest
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000     83084
           1     1.0000    0.9949    0.9975       198
           2     1.0000    1.0000    1.0000       609

    accuracy                         1.0000     83891
   macro avg     1.0000    0.9983    0.9992     83891
weighted avg     1.0000    1.0000    1.0000     83891

confusion_matrix:
 [[83084     0     0]
 [    1   197     0]
 [    0     0   609]]
{'accuracy': 0.999988079770178, 'precision': 0.9999959880443722, 'recall': 0.9983164983164983, 'f1': 0.9991541121535742, 'roc_auc': 0.9999999950284958}


243

In [ ]:
from xgboost import XGBClassifier

# scale_pos_weight is binary; for multi-class use sample_weight
xgb = XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    tree_method='hist',          # CPU
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train, sample_weight=sw)
m3, _ = eval_model('XGBoost', xgb, X_test, y_test)
joblib.dump(xgb, MODEL_DIR / 'xgboost.joblib')
(MET_DIR / 'XGBoost.json').write_text(json.dumps(m3, indent=2))

NameError: name 'sw' is not defined

In [ ]:
# sklearn GB has no class_weight; use sample_weight from inverse class freq
vc = y_train.value_counts()
w_map = {c: len(y_train) / (len(vc) * cnt) for c, cnt in vc.items()}
sw = y_train.map(w_map)

gb = GradientBoostingClassifier(
    random_state=42,
    max_depth=3,
    learning_rate=0.08,
    n_estimators=150
)
gb.fit(X_train, y_train, sample_weight=sw)
m2, _ = eval_model('Gradient_Boosting', gb, X_test, y_test)
joblib.dump(gb, MODEL_DIR / 'gradient_boosting.joblib')
(MET_DIR / 'Gradient_Boosting.json').write_text(json.dumps(m2, indent=2))


 Gradient_Boosting
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000     83084
           1     1.0000    0.9949    0.9975       198
           2     1.0000    1.0000    1.0000       609

    accuracy                         1.0000     83891
   macro avg     1.0000    0.9983    0.9992     83891
weighted avg     1.0000    1.0000    1.0000     83891

confusion_matrix:
 [[83084     0     0]
 [    1   197     0]
 [    0     0   609]]
{'accuracy': 0.999988079770178, 'precision': 0.9999959880443722, 'recall': 0.9983164983164983, 'f1': 0.9991541121535742, 'roc_auc': 0.9999998698241458}


247

In [ ]:
from xgboost import XGBClassifier

# scale_pos_weight is binary; for multi-class use sample_weight
xgb = XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    tree_method='hist',          # CPU
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train, sample_weight=sw)
m3, _ = eval_model('XGBoost', xgb, X_test, y_test)
joblib.dump(xgb, MODEL_DIR / 'xgboost.joblib')
(MET_DIR / 'XGBoost.json').write_text(json.dumps(m3, indent=2))


 XGBoost
              precision    recall  f1-score   support

           0     1.0000    0.9999    1.0000     83084
           1     0.9950    1.0000    0.9975       198
           2     0.9902    1.0000    0.9951       609

    accuracy                         0.9999     83891
   macro avg     0.9951    1.0000    0.9975     83891
weighted avg     0.9999    0.9999    0.9999     83891

confusion_matrix:
 [[83077     1     6]
 [    0   198     0]
 [    0     0   609]]
{'accuracy': 0.9999165583912458, 'precision': 0.9950729256036279, 'recall': 0.99997191597259, 'f1': 0.9975123399040786, 'roc_auc': 0.999999942082202}


235

In [ ]:
summary = [m0, m1, m2, m3]
sum_df = pd.DataFrame(summary)[['model','category','accuracy','precision','recall','f1','roc_auc']]
print(sum_df.sort_values('f1', ascending=False))

sum_path = MET_DIR / 'baselines_summary.json'
sum_path.write_text(json.dumps(summary, indent=2))
sum_df.to_csv(MET_DIR / 'baselines_summary.csv', index=False)

# also save split sizes for report
meta = {
    'data_file': str(DATA),
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'features': feature_cols,
    'label_map': {'NONE':0,'EXTEND_GREEN':1,'TRUNCATE_RED':2},
    'note': 'Trained on real Delhi bus node logs joined to GTFS; labels from rule engine on real features'
}
(MET_DIR / 'baselines_train_meta.json').write_text(json.dumps(meta, indent=2))
print('Saved:', sum_path)
print('STEP 5a OK')

                 model category  accuracy  precision    recall        f1  \
1        Random_Forest  Base ML  0.999988   0.999996  0.998316  0.999154   
2    Gradient_Boosting  Base ML  0.999988   0.999996  0.998316  0.999154   
3              XGBoost  Base ML  0.999917   0.995073  0.999972  0.997512   
0  Logistic_Regression  Base ML  0.965098   0.510222  0.983807  0.602992   

   roc_auc  
1  1.00000  
2  1.00000  
3  1.00000  
0  0.99365  
Saved: /content/drive/MyDrive/TrafficAI_ControlCenter/metrics/module1/baselines_summary.json
STEP 5a OK


In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

True Tesla T4
